In [ ]:

# Generation of 150 handwritten seeds across all 5 tasks.
# Evaluation datasets are strictly kept out of this notebook.

import os, json, random, hashlib, datetime, csv, re
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

PROJECT = Path('/content/drive/MyDrive/amharic_distillation')
PROJECT.mkdir(parents=True, exist_ok=True)
for d in ['data', 'logs', 'adapter', 'shards']:
    (PROJECT / d).mkdir(exist_ok=True)

SEED = 42
random.seed(SEED)

log_path = PROJECT / 'logs' / 'LOG.md'
if not log_path.exists():
    log_path.write_text(
        f"# Distillation Log\n\nStarted: {datetime.datetime.now().isoformat()}\nSeed: {SEED}\n\n"
    )
print(f"Project root: {PROJECT}")

Mounted at /content/drive
Project root: /content/drive/MyDrive/amharic_distillation


In [ ]:
#  Instruction templates (8-10 per task)
# Vary masculine/feminine/polite: ስጥ / ስጭ / ይስጡ / ስጠኝ
# Completely independent wordings.

INSTRUCTION_TEMPLATES = {
    "closed_book_knowledge": [
        "ለሚከተለው ጥያቄ መልስ ስጥ።",
        "ለሚከተለው ጥያቄ መልስ ስጭ።",
        "ለሚከተለው ጥያቄ መልስ ይስጡ።",
        "ጥያቄውን መልስ።",
        "ለሚከተለው ጥያቄ አጭር መልስ ስጠኝ።",
        "በአጭሩ ለሚከተለው ጥያቄ ምላሽ ስጥ።",
        "ለዚህ ጥያቄ መልስ ስጥ።",
        "ለሚከተለው ጥያቄ ምላሽ ስጭ።",
    ],
    "instruction_following": [
        "ቀጣይ የተሰጠውን ምንባብ ካነበብህ በኋላ ለጥያቄው መልስ ስጥ።",
        "ቀጣይ የተሰጠውን ምንባብ ካነበብሽ በኋላ ለጥያቄው መልስ ስጭ።",
        "ከታች ያለውን ምንባብ ካነበብክ በኋላ ጥያቄውን መልስ።",
        "ምንባቡን አንብበህ ጥያቄውን መልስ ስጥ።",
        "የተሰጠውን ጽሑፍ መሠረት በማድረግ ለጥያቄው መልስ ስጥ።",
        "ከሚከተለው ምንባብ ጋር ተያያዥ ለሆነው ጥያቄ መልስ ስጭ።",
        "ምንባቡን ተመልከትና ለጥያቄው ምላሽ ስጥ።",
        "የሚከተለውን ምንባብ አንብበህ ለጥያቄው ምላሽ ስጠኝ።",
        "ጽሑፉን ካነበብክ በኋላ ጥያቄውን ከጽሑፉ ውስጥ መልስ።",
    ],
    "summarization": [
        "ለሚከተለው ጽሁፍ ማጠቃለያ አውጣለት።",
        "ለሚከተለው ጽሁፍ ማጠቃለያ አውጪለት።",
        "ለሚከተለው ጽሁፍ ማጠቃለያ ያውጡለት።",
        "ጽሑፉን አጠቃለህ ግለጽ።",
        "ከታች ያለውን ጽሑፍ ጠቅለል አድርገህ ግለጽ።",
        "ለሚከተለው ጽሁፍ ጠቅለል ያለ ማጠቃለያ ስጥ።",
        "ይህንን ጽሑፍ በአጭሩ ጠቅለል አድርገህ ፃፍ።",
        "ጽሑፉን አንብበህ ማጠቃለያውን ፃፍ።",
    ],
    "translation_en_am": [
        "ከ English ወደ አማርኛ ተርጉም።",
        "ከ English ወደ አማርኛ ተርጉሚ።",
        "ከ English ወደ አማርኛ ይተርጉሙ።",
        "ከእንግሊዝኛ ወደ አማርኛ ተርጉም።",
        "ከእንግሊዝኛ ወደ አማርኛ ተርጉሜልኝ።",
        "ወደ አማርኛ ተርጉም።",
        "ይህንን ዓረፍተ ነገር ወደ አማርኛ ለውጥ።",
        "የሚከተለውን ወደ አማርኛ ተርጉም።",
    ],
    "translation_am_en": [
        "ከ አማርኛ ወደ English ተርጉም።",
        "ከ አማርኛ ወደ English ተርጉሚ።",
        "ከ አማርኛ ወደ English ይተርጉሙ።",
        "ከአማርኛ ወደ እንግሊዝኛ ተርጉም።",
        "ከአማርኛ ወደ እንግሊዝኛ ተርጉሜልኝ።",
        "ወደ English ተርጉም።",
        "ይህንን ዓረፍተ ነገር ወደ English ለውጥ።",
        "የሚከተለውን ወደ English ተርጉም።",
    ],
}

for task, t in INSTRUCTION_TEMPLATES.items():
    print(f"{task}: {len(t)} templates")



closed_book_knowledge: 8 templates
instruction_following: 9 templates
summarization: 8 templates
translation_en_am: 8 templates
translation_am_en: 8 templates


In [ ]:
# Hand-written seeds (30 per task) - used this at first , but later used sources from AddisGPT,Aya and other HF datasets
# 100% independent custom seeds across all five tasks.

SEEDS_CLOSED_BOOK = [
    "የአክሱም ስልጣኔ በምን ይታወቃል?",
    "ንግሥተ ዘውዲቱ መቼ ነገሡ?",
    "የአድዋ ጦርነት የተካሄደው መቼ ነው?",
    "ላሊበላ ውስጥ ያሉት ውቅር አብያተ ክርስቲያናት ስንት ናቸው?",
    "ኢትዮጵያ ለመጀመሪያ ጊዜ ኦሎምፒክ የተሳተፈችው መቼ ነው?",
    "ወባ በምን አይነት ትንኝ ይተላለፋል?",
    "ቫይታሚን ሲ ለሰውነታችን ለምን ያስፈልጋል?",
    "የስኳር በሽታ ምልክቶች ምንድናቸው?",
    "ክትባት ለህፃናት ለምን አስፈላጊ ነው?",
    "ንጹህ ውሃ ማግኘት ለጤና ለምን ወሳኝ ነው?",
    "ፀሐይ ከምድር ምን ያህል ትርቃለች?",
    "ፎቶሲንተሲስ ምንድን ነው?",
    "ውሃ በስንት ዲግሪ ሴልሺየስ ይፈላል?",
    "ዲ ኤን ኤ ምንድን ነው? ለምን ያስፈልጋል?",
    "የስበት ሕግ ማን ፈጠረው?",
    "ቡና ለኢትዮጵያ ኢኮኖሚ ምን ያህል አስፈላጊ ነው?",
    "ጤፍ ምንድን ነው? ለምን ይመረታል?",
    "በኢትዮጵያ ዋና ዋና የሰብል ዓይነቶች ምንድናቸው?",
    "ሰው ሰራሽ አስተውሎት (AI) ምንድን ነው?",
    "ኢንተርኔት እንዴት ይሰራል?",
    "ስማርት ስልክ ለትምህርት እንዴት ይጠቅማል?",
    "እንጀራ ከምን ይሠራል?",
    "ብር ለምን ተባለ? ስያሜው ከየት መጣ?",
    "በኢትዮጵያ ያለው የቀን አቆጣጠር ከግሪጎሪያን ቀን አቆጣጠር በምን ይለያል?",
    "ሰሜን ተራሮች ብሔራዊ ፓርክ የት ይገኛል?",
    "ጣና ሐይቅ ስንት ደሴቶች አሉት?",
    "አባይ ወንዝ ከየት ይነሳል?",
    "ጥምቀት በዓል በኢትዮጵያ እንዴት ይከበራል?",
    "ፍቅር እስከ መቃብር መጽሐፍ ማን ፃፈው?",
    "የኢትዮጵያ ቡና ሥነ ሥርዓት ምንድን ነው?",
]

SEEDS_INSTRUCTION_FOLLOWING = [
    "በጽሑፉ ውስጥ የተጠቀሰው ዋና ችግር ምንድን ነው?",
    "ጸሐፊው ምን ዓይነት መፍትሄ ያቀርባል?",
    "በጽሑፉ መሠረት ዋናው ምክንያት ምንድን ነው?",
    "ይህ ክስተት የተከሰተው መቼ ነው?",
    "በጽሑፉ ውስጥ ስንት ሰዎች ተጠቅሰዋል?",
    "ጽሑፉ የሚያነሳው ዋና ርዕሰ ጉዳይ ምንድን ነው?",
    "ደራሲው ስለ ትምህርት ምን ይላል?",
    "በጽሑፉ ውስጥ የተጠቀሰው ቦታ የት ነው?",
    "ይህ ፕሮጀክት ለማን ይጠቅማል?",
    "ጽሑፉ ምን ዓይነት ማስረጃ ያቀርባል?",
    "በጽሑፉ መሠረት ውጤቱ ምን ሆነ?",
    "የተገለጸው ችግር ምን ያህል ሰዎች ይነካል?",
    "ጸሐፊው ለምን ይህንን ሃሳብ ይቃወማል?",
    "በጽሑፉ ውስጥ ያለው ዋና ክርክር ምንድን ነው?",
    "ይህ ለውጥ ምን ዓይነት ተጽዕኖ አለው?",
    "በጽሑፉ ውስጥ የተጠቀሰው ድርጅት ምንድን ነው?",
    "ጽሑፉ ስለ ምን ጉዳይ ይናገራል?",
    "ደራሲው ምን ዓይነት ምክር ይሰጣል?",
    "ይህ ክስተት ለማን ጠቃሚ ነው?",
    "በጽሑፉ መሠረት ዋናው ውጤት ምንድን ነው?",
    "ጸሐፊው ለምን በዛ ድምዳሜ ላይ ደረሰ?",
    "ጽሑፉ ስለ ኢትዮጵያ ምን ይገልጻል?",
    "በተሰጠው ምንባብ ውስጥ ዋናው ገጸ ባሕሪ ማን ነው?",
    "ጸሐፊው ስለ አካባቢ ጥበቃ ምን ይመክራል?",
    "ይህ መረጃ ለማህበረሰቡ ምን ይፈይዳል?",
    "ችግሩ ለምን ተባባሰ?",
    "ጽሑፉ ስለ ጤና ምን ይገልጻል?",
    "በምንባቡ ውስጥ ያለው ቁልፍ ቁጥር ምንድን ነው?",
    "ይህ ፕሮግራም እንዴት ይተገበራል?",
    "ጸሐፊው ለወደፊት ምን ይጠብቃል?",
]

SEEDS_SUMMARIZATION = [
    "የላሊበላ ውቅር አብያተ ክርስቲያናት በ12ኛው ክፍለ ዘመን በንጉሥ ላሊበላ ዘመነ መንግሥት የተገነቡ አስደናቂ ቅርሶች ናቸው። እነዚህ ከአንድ ወጥ ድንጋይ የተቀረጹ አስራ አንድ አብያተ ክርስቲያናት የኢትዮጵያ ጥንታዊ ስነ-ህንጻ እና የሃይማኖት ጥበብ ቁንጮ ተደርገው ይቆጠራሉ። ዩኔስኮ በዓለም ቅርስነት የመዘገባቸው ሲሆን በየዓመቱ በርካታ የውጭ እና የሀገር ውስጥ ጎብኚዎችን ይስባሉ። ይሁን እንጂ የተፈጥሮ የአየር ንብረት ለውጥ እና የዝናብ ውሃ መሸርሸር በቅርሶቹ ላይ አደጋ እንዳያደርሱ ጥንቃቄ የተሞላበት የእድሳት ስራ ያስፈልጋቸዋል።",
    "ቡና የኢትዮጵያ ዋነኛ የወጪ ንግድ ምርት ሲሆን ለሚሊዮኖች ዜጎች የገቢ ምንጭ ነው። ሀገሪቱ የአረቢካ ቡና መገኛ እንደመሆኗ መጠን ልዩ መዓዛ እና ጣዕም ያላቸው የቡና ዝርያዎች በዓለም አቀፍ ገበያ ተፈላጊነት አላቸው። በቅርብ ዓመታት የቡና አምራች ገበሬዎች በቀጥታ ወደ ውጭ ገበያ እንዲልኩ የሚያስችሉ የህግ ማሻሻያዎች ተደርገዋል። ይህም አርሶ አደሩ ተገቢውን የዋጋ ተጠቃሚነት እንዲያገኝ እና ጥራቱን እንዲያሻሽል አበረታቶታል።",
    "የአየር ንብረት ለውጥ በምስራቅ አፍሪካ ሀገራት ላይ በተደጋጋሚ ድርቅ እና ያልተጠበቀ ጎርፍ እያስከተለ ይገኛል። በተለይም በግብርናው ዘርፍ ላይ የሚተዳደሩ የገጠር ማህበረሰቦች የምርት መቀነስ እና የምግብ ዋስትና እጦት ያጋጥማቸዋል። ይህንን ለመቋቋም ድርቅን የሚቋቋሙ የሰብል ዝርያዎችን መጠቀም እና የአፈር እርጥበትን የሚጠብቁ የግብርና ዘዴዎችን ማስፋፋት ወሳኝ መፍትሔ ነው። በተጨማሪም የዛፍ ተከላ እና የተፋሰስ ልማት ስራዎችን ማጠናከር አካባቢን መልሶ ለማቋቋም ይረዳል።",
    "የኢትዮጵያ አየር መንገድ በአፍሪካ አህጉር ፈር ቀዳጅ እና ግዙፍ የአየር ትራንስፖርት ተቋም ነው። ዘመናዊ አውሮፕላኖችን በመጠቀም ከአፍሪካ ወደ ሌሎች አህጉራት ፈጣን እና አስተማማኝ የበረራ ትስስር በመፍጠር ይታወቃል። አየር መንገዱ የመንገደኞች ብቻ ሳይሆን የጭነት አውሮፕላን አገልግሎቱን በማስፋፋት የሀገሪቱን የንግድ እንቅስቃሴ በእጅጉ ደግፏል። የአቪዬሽን አካዳሚውም ለአህጉሪቱ ወጣቶች የሙያ ስልጠና በመስጠት የሰው ኃይል እያፈራ ይገኛል።",
    "የደን መጨፍጨፍ የአፈር መሸርሸርን በማስከተል የመሬት ለምነት እንዲቀንስ ከሚያደርጉ ዋና ዋና ምክንያቶች አንዱ ነው። ዛፎች ሲቆረጡ የአፈር የላይኛው ክፍል በዝናብ እና በነፋስ በቀላሉ ተጠርጎ ይሄዳል። ይህንን አደጋ ለመግታት የተጀመሩ የችግኝ ተከላ ዘመቻዎች በየዓመቱ ሚሊዮኖች ችግኞችን በመትከል የተራቆቱ ተራሮችን መልሶ ለማልበስ አግዘዋል። የህብረተሰቡ ተሳትፎ እና የተተከሉትን ችግኞች የመንከባከብ ባህል ማደጉም ለዘላቂነቱ ወሳኝ ነው።",
    "ንጹህ የመጠጥ ውሃ ለሰው ልጅ ጤና እና ደህንነት መሠረታዊ ፍላጎት ነው። በቂ እና ንጹህ ውሃ ባለመኖሩ ምክንያት የሚከሰቱ የውሃ ወለድ በሽታዎች በህፃናት ጤና ላይ ከፍተኛ ጉዳት ያደርሳሉ። መንግሥታዊ እና መንግሥታዊ ያልሆኑ ድርጅቶች በገጠር መንደሮች የውሃ ጉድጓዶችን በመቆፈር የማህበረሰቡን ጫና እያቃለሉ ነው። ንጹህ ውሃ ማግኘት ሴቶችና ልጆች ውሃ ለመቅዳት የሚያጠፉትን ረጅም የእግር ጉዞ በማስቀረት ለትምህርትና ለስራ ጊዜ እንዲያገኙ ያስችላቸዋል።",
    "የዲጂታል ክፍያ ስርዓት በኢትዮጵያ የንግድ እና የባንክ አገልግሎት ውስጥ አዲስ ምዕራፍ ከፍቷል። በሞባይል ስልክ አማካኝነት ገንዘብ መላክ እና የዕለት ተዕለት ግብይቶችን መፈጸም የጥሬ ገንዘብ ፍላጎትን በእጅጉ ቀንሶታል። ይህም የገንዘብ ዝውውሩን ፈጣን፣ አስተማማኝ እና ወጪ ቆጣቢ እንዲሆን አስችሎታል። በተለይም የባንክ ቅርንጫፎች በሌሉባቸው የገጠር አካባቢዎች የፋይናንስ ተደራሽነትን በማስፋፋት የህዝቡን የኢኮኖሚ ተሳትፎ አሳድጓል።",
    "ጤፍ በኢትዮጵያ ብቻ በስፋት የሚመረት እና ለሀገሪቱ ዋና ምግብ የሆነው የእህል ዓይነት ነው። ከቅርብ ጊዜ ወዲህ ከግሉተን ነፃ በመሆኑ እና ከፍተኛ የብረትና የካልሲየም ይዘት ስላለው በዓለም አቀፍ ደረጃ ከፍተኛ ተወዳጅነትን አትርፏል። በዓለም ዙሪያ ያሉ የስነ-ምግብ ተመራማሪዎች ጤፍን እንደ ድንቅ አልሚ ምግብ ይመድቡታል። የጤፍን ምርታማነት በዘመናዊ ቴክኖሎጂ ማሳደግ ለሀገር ውስጥ የምግብ ዋስትናም ሆነ ለወጪ ንግድ ትልቅ አስተዋጽኦ አለው።",
    "የጥምቀት በዓል በኢትዮጵያ በየዓመቱ ጥር 11 ቀን በታላቅ ድምቀት የሚከበር ሃይማኖታዊ እና ባህላዊ በዓል ነው። ታቦታቱ ከየአድባራቱ ወጥተው ወደ ባሕረ ጥምቀቱ ሲያቀኑ በሚሊዮን የሚቆጠሩ ምእመናን በባህላዊ ነጭ ልብሶች አጊጠው ይከተሏቸዋል። ይህ በዓል የሰላም፣ የፍቅር እና የአብሮነት መገለጫ ተደርጎ የሚወሰድ ሲሆን በዩኔስኮ በማይዳሰስ ቅርስነት ተመዝግቧል። በርካታ የውጭ ሀገር ጎብኚዎችም የበዓሉን ድምቀት ለመታደም ወደ ጎንደር እና አዲስ አበባ ይመጣሉ።",
    "ታላቁ የኢትዮጵያ ህዳሴ ግድብ በአባይ ወንዝ ላይ እየተገነባ ያለ ግዙፍ የኤሌክትሪክ ኃይል ማመንጫ ፕሮጀክት ነው። ግድቡ የሀገሪቱን የኃይል አቅርቦት በእጥፍ በማሳደግ የኢንዱስትሪ ልማትን እና የከተሞችን እድገት ለማፋጠን ያለመ ነው። በተጨማሪም ለጎረቤት ሀገራት ንጹህ ኃይል በመሸጥ የቀጠናውን የኢኮኖሚ ትስስር ያጠናክራል ተብሎ ይጠበቃል። ፕሮጀክቱ በህዝብ እና በመንግሥት ጥምር አቅም እየተገነባ ያለ የኢኮኖሚ ሉዓላዊነት ምልክት ነው።",
    "ወባ በትንኝ ንክሻ የሚተላለፍ አደገኛ በሽታ ሲሆን በተለይም በቆላማ እና ሞቃታማ አካባቢዎች በስፋት ይከሰታል። በሽታውን ለመከላከል በኬሚካል የተነከሩ የአልጋ አጎበሮችን በአግባቡ መጠቀም እና የትንኝ መራቢያ የሆኑ የቆሙ ውሃዎችን ማድረቅ ያስፈልጋል። የጤና ኤክስቴንሽን ሰራተኞች በማህበረሰቡ ውስጥ ግንዛቤ በመፍጠር እና ፈጣን ምርመራዎችን በማድረግ የበሽታውን ስርጭት ለመግታት ይሰራሉ። ቅድመ መከላከል እና ፈጣን ህክምና ማግኘት የሞት ምጣኔውን በእጅጉ ይቀንሳል።",
    "የሴቶች ትምህርት ለማህበረሰብ ሁለንተናዊ እድገት መሠረት መሆኑ በበርካታ ጥናቶች ተረጋግጧል። የተማሩ ሴቶች ጤናማ ቤተሰብ የመመስረት፣ የልጆቻቸውን ትምህርት የመከታተል እና በኢኮኖሚ ራሳቸውን የመቻል እድላቸው ሰፊ ነው። የሴት ልጆችን የትምህርት ቤት ተሳትፎ ለማሳደግ የሚደረጉ ድጋፎች ያለዕድሜ ጋብቻን ለመቀነስ ጉልህ ሚና ይጫወታሉ። ሴቶች በውሳኔ ሰጪነት ቦታዎች ላይ ያላቸው ተሳትፎ ማደጉም ለፍትሃዊ ማህበረሰብ ግንባታ ወሳኝ ነው።",
    "የሰሜን ተራሮች ብሔራዊ ፓርክ በዓለም ላይ በየትኛውም ቦታ የማይገኙ እንደ ዋሊያ አይቤክስ እና ቀይ ቀበሮ ያሉ ብርቅዬ እንስሳት መኖሪያ ነው። የተራሮቹ የተፈጥሮ ውበት እና ጥልቅ ገደሎች ለተፈጥሮ ወዳዶች አስደናቂ እይታን ይፈጥራሉ። ይሁን እንጂ የህዝብ ቁጥር መጨመር እና የመኖሪያ ቦታዎች መስፋፋት የዱር እንስሳቱን የተፈጥሮ መኖሪያ እየተጋፋው ይገኛል። የፓርኩን ህልውና ለመጠበቅ ማህበረሰቡን ያሳተፈ የጥበቃ ስራ ማከናወን አስፈላጊ ሆኗል።",
    "የቴሌሜዲሲን አገልግሎት በዲጂታል ቴክኖሎጂ አማካኝነት ታካሚዎች ከሩቅ ሆነው ከሀኪሞች ጋር የሚገናኙበት አሰራር ነው። ይህ አሰራር በገጠር የሚኖሩ ዜጎች ወደ ትላልቅ ከተሞች ሳይጓዙ የህክምና ምክር እና ማዘዣ እንዲያገኙ ያስችላቸዋል። በሀገሪቱ የኢንተርኔት ተደራሽነት እያደገ መምጣቱ ለዚህ አገልግሎት መስፋፋት ምቹ ሁኔታን ፈጥሯል። የጤና ባለሙያዎችም በበይነመረብ አማካኝነት የልምድ ልውውጥ በማድረግ የሙያ ብቃታቸውን እንዲያሳድጉ ይረዳል።",
    "ባህላዊ የሙዚቃ መሳሪያዎች እንደ ክራር፣ ማሲንቆ እና ዋሽንት የኢትዮጵያ የኪነ-ጥበብ መገለጫዎች ናቸው። እነዚህ መሳሪያዎች የህዝቡን ደስታ፣ ሀዘን፣ ታሪክ እና የፍቅር ስሜት በዜማ ለመግለጽ ለዘመናት ሲያገለግሉ ቆይተዋል። አሁንም ድረስ በዘመናዊ የሙዚቃ ቅንብሮች ውስጥ እየተካተቱ ልዩ ሀገራዊ ጣዕም እንዲኖራቸው በማድረግ ላይ ይገኛሉ። ይህንን ቅርስ ለቀጣዩ ትውልድ ለማስተላለፍ የሙዚቃ ትምህርት ቤቶችና ወጣቶች ትኩረት ሰጥተው ሊሰሩበት ይገባል።",
    "የአፈር መሸርሸርን ለመከላከል እርከን መስራት በደጋማ የኢትዮጵያ ክፍሎች ለዘመናት ሲሰራበት የቆየ ዘዴ ነው። እርከን የዝናብ ውሃ በፍጥነት እንዳይወርድ በማዘግየት የአፈሩን ንጥረ ነገር ይጠብቃል። ገበሬዎች በማህበር ተደራጅተው በተራቆቱ ኮረብቶች ላይ ድንጋይ በማንሳት እርከኖችን ሲሰሩ ይታያሉ። ይህ ስራ የመሬት ምርታማነት እንዲጨምር እና የከርሰ ምድር ውሃ እንዲጎለብት በማድረግ ትልቅ አስተዋጽኦ ያበረክታል።",
    "የንብ ማነብ ስራ አነስተኛ መሬት እና ካፒታል ላላቸው የገጠር አርሶ አደሮች ተጨማሪ ገቢ የሚያስገኝ ዘርፍ ነው። ኢትዮጵያ በተፈጥሮ እፅዋት የታደለች በመሆኗ ከፍተኛ ጥራት ያለው የተፈጥሮ ማር የማምረት አቅም አላት። ዘመናዊ የንብ ቀፎዎችን እና የማር መጭመቂያ ቴክኖሎጂዎችን መጠቀም የምርት መጠኑን በከፍተኛ ደረጃ ያሳድገዋል። የማር እና የሰም ምርቶችን በማቀነባበር ለሀገር ውስጥ ገበያም ሆነ ለውጭ ሀገራት ማቅረብ ይቻላል።",
    "ስፖርት እና የሰውነት እንቅስቃሴ ለጤናማ አኗኗር እና ለአእምሮ እድገት ወሳኝ ሚና ይጫወታሉ። አዘውትሮ የአካል ብቃት እንቅስቃሴ ማድረግ እንደ ደም ግፊት እና ስኳር ያሉ ተላላፊ ያልሆኑ በሽታዎችን ለመከላከል ይረዳል። በተጨማሪም ስፖርት የወጣቶችን ስነ-ምግባር በማነጽ እና ከሱስ በማራቅ ጠንካራ ማህበረሰብ ለመገንባት ያግዛል። በከተሞች የመጫወቻ ሜዳዎችን እና የስፖርት ማዘውተሪያ ስፍራዎችን ማስፋፋት የህዝቡን ጤና ለማሻሻል አስፈላጊ ነው።",
    "በከተሞች ፈጣን እድገት ምክንያት የሚፈጠረው የደረቅ ቆሻሻ ክምችት የአካባቢ ብክለትን እና የጤና ችግሮችን እያስከተለ ነው። ቆሻሻን በአግባቡ የመሰብሰብ እና መልሶ ጥቅም ላይ የማዋል (ሪሳይክል) ስራ ገና በጅምር ደረጃ ላይ ይገኛል። ፕላስቲኮችን እና ኦርጋኒክ ቆሻሻዎችን ለይቶ ማሰባሰብ ለአካባቢ ጽዳት ብቻ ሳይሆን ለስራ እድል ፈጠራም ይጠቅማል። ዘመናዊ የቆሻሻ አወጋገድ ስርዓት መዘርጋት ለከተሞች ውበት እና ለህዝብ ጤና ቁልፍ ተግባር ነው።",
    "የሀገር ባህል ልብስ ስራ ከጥጥ ፈታይነት ጀምሮ እስከ ሽመና እና ጥልፍ ድረስ በርካታ ባለሙያዎችን የሚያሳትፍ ጥበብ ነው። የጥበብ ቀሚሶች እና ጋቢዎች በኢትዮጵያውያን ዘንድ በበዓላት እና በሰርግ ስነ-ስርዓቶች ላይ በኩራት ይለበሳሉ። በቅርብ ጊዜ ወጣት ዲዛይነሮች ባህላዊውን ጥበብ ከዘመናዊ ፋሽን ጋር በማዋሃድ በዓለም አቀፍ መድረኮች እያስተዋወቁት ይገኛሉ። ይህ ዘርፍ የባህል መገለጫ ከመሆኑም ባሻገር ለብዙ ዜጎች የስራ እድል ፈጥሯል።",
    "የህፃናት ክትባት ገዳይ እና አካል ጉዳተኛ ከሚያደርጉ ተላላፊ በሽታዎች ልጆችን ለመጠበቅ አስተማማኝ ጋሻ ነው። እንደ ኩፍኝ፣ ፖሊዮ እና ቴታነስ ያሉ በሽታዎችን በክትባት አማካኝነት ሙሉ በሙሉ መቆጣጠር ይቻላል። መንግሥት በየጊዜው የሚያካሂዳቸው የክትባት ዘመቻዎች የህፃናት ሞት ምጣኔን በከፍተኛ ሁኔታ እንዲቀንስ አድርገዋል። ወላጆች ልጆቻቸውን በወቅቱ ወደ ጤና ጣቢያ ወስደው ማስከተብ የህፃናቱን የወደፊት ህይወት ዋስትና ይሰጣል።",
    "የፀሐይ ኃይል (ሶላር) በኢትዮጵያ ገጠር አካባቢዎች ለኤሌክትሪክ አቅርቦት አማራጭ እና ተመራጭ የኃይል ምንጭ እየሆነ ነው። የብሔራዊ የኤሌክትሪክ መስመር ባልደረሰባቸው መንደሮች የሶላር ፓነሎች ለብርሃን፣ ለስልክ ባትሪ መሙላት እና ለውሃ ፓምፕ ስራ ያገለግላሉ። ይህ ቴክኖሎጂ የጋዝ ጭስ እና የባትሪ ወጪን በማስቀረት የህዝቡን ኑሮ አቅልሏል። ታዳሽ እና ንጹህ ኃይል መጠቀም የአካባቢ ብክለትን ለመከላከልም ከፍተኛ ጠቀሜታ አለው።",
    "የሰብል ልውውጥ (ክሮፕ ሮቴሽን) በአንድ ማሳ ላይ የተለያዩ የሰብል ዓይነቶችን በየወቅቱ በማፈራረቅ የመዝራት ዘዴ ነው። በተለይም እንደ አተር እና ባቄላ ያሉ ጥራጥሬዎችን መዝራት ናይትሮጅንን ወደ አፈር በመመለስ ለምነቱን ያድሳል። ይህ ዘዴ የኬሚካል ማዳበሪያ ፍላጎትን ከመቀነሱም በላይ የተባዮች እና የበሽታዎች ስርጭት እንዳይስፋፋ ያደርጋል። አርሶ አደሮች ይህንን ልምድ በመከተል ዘላቂ የምርት እድገት ማስመዝገብ ይችላሉ።",
    "የአዲስ አበባ ቀላል ባቡር ትራንስፖርት በከተማዋ የህዝብ ትራንስፖርት ችግርን ለማቃለል የተገነባ ዘመናዊ መሰረተ ልማት ነው። በየቀኑ በሺዎች የሚቆጠሩ ሰራተኞችን እና ተማሪዎችን ከቦታ ወደ ቦታ በማጓጓዝ አገልግሎት ይሰጣል። በኤሌክትሪክ የሚሰራ በመሆኑ የአየር ብክለትን ለመቀነስ እና የከተማዋን የትራፊክ መጨናነቅ ለማቃለል ይረዳል። ይሁን እንጂ የጥገና እና የተለዋጭ እቃዎች አቅርቦትን በማሻሻል የአገልግሎት አሰጣጡን ይበልጥ ማሳደግ ያስፈልጋል።",
    "የታሪክ እና የባህል ቅርሶችን በአግባቡ ጠብቆ ማቆየት የአንድን ሀገር ማንነት እና ትውፊት ለቀጣዩ ትውልድ ለማስተላለፍ ወሳኝ ነው። በሙዚየሞች ውስጥ የሚገኙ ጥንታዊ ጽሑፎች፣ አልባሳት እና የጦር መሳሪያዎች የታሪክ ምስክሮች ናቸው። ቅርሶቹ በአግባቡ ተመዝግበው እና በዘመናዊ ሳይንሳዊ ዘዴዎች ተጠብቀው ካልተያዙ በተፈጥሮ እና በሰው ሰራሽ አደጋዎች ሊወድሙ ይችላሉ። ህብረተሰቡ ቅርሶችን የመንከባከብ እና የማስጠበቅ ሀገራዊ ኃላፊነት አለበት።",
    "የአእምሮ ጤና እንደ አካል ጤና ሁሉ እኩል ትኩረት እና እንክብካቤ የሚሻ የጤና ዘርፍ ነው። ጭንቀት፣ ድብርት እና የስነ-ልቦና ጫናዎች በሰው ልጅ የዕለት ተዕለት ህይወት እና የስራ ምርታማነት ላይ አሉታዊ ተጽዕኖ ያሳድራሉ። በማህበረሰቡ ውስጥ በአእምሮ ጤና ችግሮች ዙሪያ ያለውን የተሳሳተ አመለካከት ማረም እና ግንዛቤ መፍጠር ያስፈልጋል። በጤና ተቋማት የስነ-ልቦና ምክር እና የህክምና አገልግሎቶችን ማስፋፋት የዜጎችን ደህንነት ያረጋግጣል።",
    "የመስኖ ግብርና በዝናብ ላይ ብቻ ጥገኛ የሆነውን ባህላዊ የግብርና ስርዓት ወደ ምርታማ እና ቀጣይነት ያለው ልማት ለመቀየር ቁልፍ ነው። ወንዞችን እና የከርሰ ምድር ውሃዎችን በመጠቀም በዓመት ሁለት እና ሦስት ጊዜ ማምረት ይቻላል። በተለይም የአትክልት እና የፍራፍሬ ምርቶችን በመስኖ በማልማት የገበሬዎችን ገቢ ማሳደግ እና የከተሞችን የምግብ አቅርቦት ማረጋጋት ተችሏል። ዘመናዊ የመስኖ ቴክኖሎጂዎችን ለገበሬዎች ተደራሽ ማድረግ ትኩረት የሚሻ ጉዳይ ነው።",
    "የትምህርት ቤቶች የዲጂታል ቤተ-መጽሐፍት ማስፋፊያ ተማሪዎች የተለያዩ የማጣቀሻ መጻሕፍትን እና የምርምር ጽሑፎችን በቀላሉ እንዲያገኙ ያስችላቸዋል። በኮምፒውተር እና በታብሌቶች የሚቀርቡ የመማሪያ ቁሳቁሶች የወረቀት እጥረትን በመቅረፍ ጥራት ያለው ትምህርት ይሰጣሉ። መምህራንም ዘመናዊ የማስተማሪያ ዘዴዎችን በመጠቀም የተማሪዎቻቸውን ግንዛቤ ለማሳደግ ይችላሉ። ይህንን ቴክኖሎጂ በገጠር ትምህርት ቤቶችም ጭምር ተደራሽ ማድረግ የትምህርት ጥራትን ያጎለብታል።",
    "ባህላዊ የግጭት አፈታት ስርዓቶች በኢትዮጵያ የተለያዩ ብሔረሰቦች ዘንድ ለዘመናት ማህበራዊ ሰላምን ሲያስጠብቁ የቆዩ እሴቶች ናቸው። የሀገር ሽማግሌዎች እና የማህበረሰብ መሪዎች ተቀራርበው የሚያደርጓቸው እርቀ ሰላሞች ዘላቂ እርቅን ያሰፍናሉ። እነዚህ ስርዓቶች ውድ እና ረጅም የፍርድ ቤት ሂደቶችን በማስቀረት ወንድማማችነትን እና ይቅርታን ያበረታታሉ። ባህላዊ እሴቶችን ከዘመናዊ የፍትህ ስርዓት ጋር በማቀናጀት መጠቀም ለማህበረሰብ መረጋጋት ከፍተኛ ፋይዳ አለው።",
    "የዓሣ ሀብት ልማት በኢትዮጵያ ሐይቆች እና ወንዞች ውስጥ ሰፊ የምግብ እና የኢኮኖሚ አቅም ያለው ዘርፍ ነው። እንደ ጣና፣ ጫሞ እና ዝዋይ ባሉ ሐይቆች የሚካሄደው የዓሣ ማጥመድ ስራ ለአካባቢው ነዋሪዎች የገቢ ምንጭ ሆኗል። ዓሣ ከፍተኛ የፕሮቲን ይዘት ያለው በመሆኑ ለህብረተሰቡ የተመጣጠነ ምግብ አቅርቦት ትልቅ ጠቀሜታ አለው። ይሁን እንጂ ከልክ ያለፈ አሳ ማጥመድን በመቆጣጠር እና የዓሣ እርባታን በማስፋፋት የሀብቱን ዘላቂነት መጠበቅ ይገባል።",
]

SEEDS_TRANSLATION_EN_AM = [
    "The Ethiopian coffee ceremony is an important cultural tradition.",
    "Access to clean water remains a challenge in many rural areas.",
    "Artificial intelligence is transforming how we work and live.",
    "Farmers should use crop rotation to maintain soil fertility.",
    "Education is the most powerful tool for reducing poverty.",
    "The Great Ethiopian Renaissance Dam is one of Africa's largest.",
    "Children need vaccinations to protect them from dangerous diseases.",
    "Solar energy can help provide electricity to remote communities.",
    "Deforestation is a major environmental problem in East Africa.",
    "Mobile banking has revolutionized financial services in Ethiopia.",
    "Traditional medicine plays a significant role in Ethiopian healthcare.",
    "The population of Addis Ababa has grown rapidly in recent decades.",
    "Young entrepreneurs are creating innovative solutions for local problems.",
    "Climate change threatens agricultural productivity across the continent.",
    "Reading books from an early age helps children develop language skills.",
    "The Ethiopian flag symbolizes peace, unity, and the diversity of its people.",
    "Sustainable farming practices can increase food production while protecting nature.",
    "Public transportation in Addis Ababa includes buses, taxis, and the light rail.",
    "Tourism contributes billions of dollars to Ethiopia's economy each year.",
    "Regular exercise and a balanced diet are essential for good health.",
    "The Simien Mountains are home to rare wildlife species found nowhere else.",
    "Investing in early childhood education yields long-term economic benefits.",
    "Internet connectivity is expanding across Ethiopia through fiber optic networks.",
    "Community health workers play a vital role in disease prevention programs.",
    "The Blue Nile Falls is one of Ethiopia's most famous natural attractions.",
    "Learning a second language improves cognitive abilities and career opportunities.",
    "Ethiopian athletes are renowned for their excellence in long-distance running.",
    "Urban planning must balance economic growth with environmental sustainability.",
    "Cooperation between nations is essential for addressing global challenges.",
    "Women's participation in governance has increased significantly in recent years.",
]

SEEDS_TRANSLATION_AM_EN = [
    "ኢትዮጵያ ከአፍሪካ ሁለተኛ ሕዝብ ብዛት ያላት ሀገር ናት።",
    "አዲስ አበባ የአፍሪካ ህብረት ዋና መቀመጫ ናት።",
    "ቡና ኢትዮጵያ ውስጥ ለመጀመሪያ ጊዜ የተገኘ ተክል ነው።",
    "ትምህርት ለሁሉም ሰው መሠረታዊ መብት ነው።",
    "ንጹህ ውሃ ማግኘት ለጤና በጣም አስፈላጊ ነው።",
    "የኢትዮጵያ ብር ባለፉት ዓመታት ዋጋው ቀንሷል።",
    "ገበሬዎች ዘመናዊ የግብርና ቴክኖሎጂ እንዲጠቀሙ ማበረታታት ያስፈልጋል።",
    "በኢትዮጵያ ውስጥ ከ80 በላይ ብሄር ብሄረሰቦች ይኖራሉ።",
    "ፀሐይ ለምድራችን ብርሃንና ሙቀት ዋና ምንጭ ናት።",
    "ዛፎችን መትከል የአየር ንብረት ለውጥን ለመቀነስ ይረዳል።",
    "ወባ በትንኝ ንክሻ የሚተላለፍ በሽታ ነው።",
    "የኢትዮጵያ አየር መንገድ በአፍሪካ ትልቁ አየር መንገድ ነው።",
    "ስፖርት ለወጣቶች ጤንነትም ለአእምሮ እድገትም ይጠቅማል።",
    "ቴክኖሎጂ ትምህርትን ለሁሉም ተደራሽ ለማድረግ ያግዛል።",
    "እንጀራ ከጤፍ ዱቄት ይሠራል፤ ዋናው የኢትዮጵያ ምግብ ነው።",
    "ምርምር ማድረግ ለሳይንስ እና ለቴክኖሎጂ እድገት ወሳኝ ነው።",
    "ሴቶች በኢኮኖሚ ውስጥ ያላቸው ድርሻ ከዓመት ዓመት እየጨመረ ነው።",
    "ባህላዊ ሙዚቃ የህብረተሰቡን ታሪክና ማንነት ይገልጻል።",
    "ለብክለት ቅነሳ ተፈጥሮ ተስማሚ ቁሳቁሶችን መጠቀም ይመከራል።",
    "ልጆች ደህንነታቸው የተጠበቀ ትምህርት ቤት ያስፈልጋቸዋል።",
    "በረሃማነት ለግብርና ተስማሚ መሬትን ያበላሻል።",
    "የኢትዮጵያ ቅርስ ቦታዎች በዓለም ቅርስ ዝርዝር ውስጥ ይገኛሉ።",
    "መንግሥት ለጤና ተቋማት በቂ ድጋፍ ማድረግ ይጠበቅበታል።",
    "ሁሉም ዜጎች ድምጻቸው መሰማት ያለበት ዴሞክራሲያዊ መብት ነው።",
    "ግድብ መገንባት ለኤሌክትሪክ ኃይል ማመንጫ ዋና መንገድ ነው።",
    "በዓለም ላይ ቢያንስ 7,000 ቋንቋዎች ይነገራሉ።",
    "ንግድ ለሀገር እድገት ሁል ጊዜ ወሳኝ ሚና ይጫወታል።",
    "ማንበብ ለአእምሮ እድገት እና ለእውቀት ማስፋት ይጠቅማል።",
    "ከተሞች ለሥራ ፍለጋ ወደ ከተማ የሚመጡ ሰዎችን ለመቅበል ዝግጁ መሆን አለባቸው።",
    "ነዳጅ ዘይት ዋጋ ከፍ ሲል የትራንስፖርት ወጪ ይጨምራል።",
]

print(f"Closed-book seeds: {len(SEEDS_CLOSED_BOOK)}")
print(f"Instruction-following seeds: {len(SEEDS_INSTRUCTION_FOLLOWING)}")
print(f"Summarization seeds: {len(SEEDS_SUMMARIZATION)}")
print(f"Translation EN->AM seeds: {len(SEEDS_TRANSLATION_EN_AM)}")
print(f"Translation AM->EN seeds: {len(SEEDS_TRANSLATION_AM_EN)}")



Closed-book seeds: 30
Instruction-following seeds: 30
Summarization seeds: 30
Translation EN->AM seeds: 30
Translation AM->EN seeds: 30


In [ ]:
seeds_data = {
    "closed_book": SEEDS_CLOSED_BOOK,
    "instruction_following": SEEDS_INSTRUCTION_FOLLOWING,
    "summarization": SEEDS_SUMMARIZATION,
    "translation_en_am": SEEDS_TRANSLATION_EN_AM,
    "translation_am_en": SEEDS_TRANSLATION_AM_EN,
    "instruction_templates": INSTRUCTION_TEMPLATES,
}


In [ ]:
with open(PROJECT / 'data' / 'seeds.json', 'w', encoding='utf-8') as f:
    json.dump(seeds_data, f, ensure_ascii=False, indent=2)
print("Saved seeds.json")

In [ ]:
#  Assemble and format all 150 seed prompts

def format_prompt(instruction, context=None, question=None):
    parts = [instruction]
    if context:
        parts.append(f"\nContext:\n{context}")
    if question:
        parts.append(f"\n{question}")
    return "\n".join(parts)

all_seed_prompts = []

# 1. Closed-book knowledge (30)
for q in SEEDS_CLOSED_BOOK:
    instr = random.choice(INSTRUCTION_TEMPLATES["closed_book_knowledge"])
    all_seed_prompts.append({
        "id": f"seed_cb_{len(all_seed_prompts):04d}",
        "task": "closed_book_knowledge",
        "direction": "",
        "instruction": instr,
        "context": q,
        "question": "",
        "prompt": format_prompt(instr, context=q),
        "source": "seed_handwritten",
    })

# 2. Instruction following (30)
for q in SEEDS_INSTRUCTION_FOLLOWING:
    instr = random.choice(INSTRUCTION_TEMPLATES["instruction_following"])
    all_seed_prompts.append({
        "id": f"seed_if_{len(all_seed_prompts):04d}",
        "task": "instruction_following",
        "direction": "",
        "instruction": instr,
        "context": "",
        "question": q,
        "prompt": format_prompt(instr, question=q),
        "source": "seed_handwritten",
    })

# 3. Summarization (30)
for passage in SEEDS_SUMMARIZATION:
    instr = random.choice(INSTRUCTION_TEMPLATES["summarization"])
    all_seed_prompts.append({
        "id": f"seed_sm_{len(all_seed_prompts):04d}",
        "task": "summarization",
        "direction": "",
        "instruction": instr,
        "context": passage,
        "question": "",
        "prompt": format_prompt(instr, context=passage),
        "source": "seed_handwritten",
    })

# 4. Translation EN->AM (30)
for sent in SEEDS_TRANSLATION_EN_AM:
    instr = random.choice(INSTRUCTION_TEMPLATES["translation_en_am"])
    all_seed_prompts.append({
        "id": f"seed_tr_en_am_{len(all_seed_prompts):04d}",
        "task": "translation",
        "direction": "en->am",
        "instruction": instr,
        "context": sent,
        "question": "",
        "prompt": format_prompt(instr, context=sent),
        "source": "seed_handwritten",
    })

# 5. Translation AM->EN (30)
for sent in SEEDS_TRANSLATION_AM_EN:
    instr = random.choice(INSTRUCTION_TEMPLATES["translation_am_en"])
    all_seed_prompts.append({
        "id": f"seed_tr_am_en_{len(all_seed_prompts):04d}",
        "task": "translation",
        "direction": "am->en",
        "instruction": instr,
        "context": sent,
        "question": "",
        "prompt": format_prompt(instr, context=sent),
        "source": "seed_handwritten",
    })

with open(PROJECT / 'data' / 'seed_prompts.json', 'w', encoding='utf-8') as f:
    json.dump(all_seed_prompts, f, ensure_ascii=False, indent=2)

print(f"Saved seed_prompts.json with {len(all_seed_prompts)} prompts.")


Saved seed_prompts.json with 150 prompts.


In [ ]:
# ── Cell 1.6: Count validation ───────────────────────────────────────
# Explicit verification of all seed counts as required by the plan.

cb_count = sum(1 for p in all_seed_prompts if p['task'] == 'closed_book_knowledge')
if_count = sum(1 for p in all_seed_prompts if p['task'] == 'instruction_following')
sm_count = sum(1 for p in all_seed_prompts if p['task'] == 'summarization')
en_am_count = sum(1 for p in all_seed_prompts if p['task'] == 'translation' and p['direction'] == 'en->am')
am_en_count = sum(1 for p in all_seed_prompts if p['task'] == 'translation' and p['direction'] == 'am->en')
total_count = len(all_seed_prompts)

print("=" * 60)
print("SEED PROMPTS COUNT VALIDATION")
print("=" * 60)
print(f"  Closed-book:           {cb_count:3d}  (expected: 30)")
print(f"  Instruction-following: {if_count:3d}  (expected: 30)")
print(f"  Summarization:         {sm_count:3d}  (expected: 30)")
print(f"  Translation EN->AM:    {en_am_count:3d}  (expected: 30)")
print(f"  Translation AM->EN:    {am_en_count:3d}  (expected: 30)")
print("-" * 60)
print(f"  Total seed prompts:    {total_count:3d}  (expected: 150)")
print("=" * 60)

assert cb_count == 30, f"Expected 30 closed-book, got {cb_count}"
assert if_count == 30, f"Expected 30 instruction-following, got {if_count}"
assert sm_count == 30, f"Expected 30 summarization, got {sm_count}"
assert en_am_count == 30, f"Expected 30 EN->AM, got {en_am_count}"
assert am_en_count == 30, f"Expected 30 AM->EN, got {am_en_count}"
assert total_count == 150, f"Expected 150 total, got {total_count}"

print("All count assertions passed successfully!")


SEED PROMPTS COUNT VALIDATION
  Closed-book:            30  (expected: 30)
  Instruction-following:  30  (expected: 30)
  Summarization:          30  (expected: 30)
  Translation EN->AM:     30  (expected: 30)
  Translation AM->EN:     30  (expected: 30)
------------------------------------------------------------
  Total seed prompts:    150  (expected: 150)
All count assertions passed successfully!


# Teacher pilot (hard gate), passage generation, prompt expansion, answer generation, eval answers.

In [ ]:
import subprocess, sys
print(subprocess.check_output(['nvidia-smi']).decode())
print(f"Python: {sys.version}")

from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
PROJECT = Path('/content/drive/MyDrive/amharic_distillation')
assert PROJECT.exists(), "Run the previous cells first!"
SEED = 42

Thu Oct  8 21:01:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 33.4 MB/s eta 0:00:00


In [ ]:
import importlib.metadata

# Modern way to check package versions without pkg_resources
for pkg in ['transformers', 'accelerate', 'bitsandbytes', 'torch', 'sentencepiece']:
    try:
        version = importlib.metadata.version(pkg)
        print(f"{pkg}: {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{pkg}: not found")

transformers: 5.18.0
accelerate: 1.15.0
bitsandbytes: 0.50.2
torch: 2.11.0+cu130
sentencepiece: 0.2.2


In [ ]:
import torch, time, gc, re, json, random
from transformers import AutoProcessor, AutoModelForMultimodalLM, BitsAndBytesConfig
random.seed(SEED)


In [ ]:

# HARD GATE: Gemma 4 12B-it was my trail option but it faiiled the pilot test and switch to Gemma 4 E4B
TEACHER_ID = "google/gemma-4-E4B-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

teacher_processor = None
teacher_model = None
TEACHER_USED = None

try:
    print(f"Attempting to load preferred candidate {TEACHER_ID} in 4-bit...")
    t0 = time.time()
    teacher_processor = AutoProcessor.from_pretrained(TEACHER_ID)
    teacher_model = AutoModelForMultimodalLM.from_pretrained(
        TEACHER_ID,
        quantization_config=bnb_config,
        device_map="auto",
        dtype=torch.float16,
    )
    teacher_model.eval()
    mem_alloc = torch.cuda.max_memory_allocated() / 1e9
    print(f"Successfully loaded {TEACHER_ID} in {time.time()-t0:.1f}s | VRAM: {mem_alloc:.2f} GB")
    TEACHER_USED = TEACHER_ID
except Exception as e:
    print(f"Failed to load {TEACHER_ID} in 4-bit: {e}")
    TEACHER_USED = None

print(f"\n>>> ACTIVE TEACHER MODEL: {TEACHER_USED}")

Attempting to load preferred candidate google/gemma-4-E4B-it in 4-bit...


processor_config.json:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/18.6k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/5.14k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.08k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 16.0GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Successfully loaded google/gemma-4-E4B-it in 288.7s | VRAM: 9.30 GB

>>> ACTIVE TEACHER MODEL: google/gemma-4-E4B-it


In [ ]:
#Generation helper + system prompts
def generate_one(model, processor, prompt_text, system_prompt,
                 max_new_tokens=300, temperature=0.4, top_p=0.9):
    """
    Generate one response using official Gemma 4 chat template pattern.
    Uses native system role and enable_thinking=False.
    Returns text + metadata for local truncation tracking.
    """
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt_text},
    ]

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=True,
        enable_thinking=False,
    ).to(model.device)

    input_len = inputs["input_ids"].shape[-1]

    torch.cuda.synchronize()
    t0 = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
        )
    torch.cuda.synchronize()
    elapsed = time.time() - t0

    new_tokens = outputs[0][input_len:]
    num_gen = len(new_tokens)

    # Decode response
    raw_response = processor.decode(new_tokens, skip_special_tokens=False)
    # Clean turn markers and whitespace
    text = raw_response.replace("<end_of_turn>", "").replace("<eos>", "").strip()
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()

    # Determine EOS and truncation locally
    eos_id = getattr(processor, "eos_token_id", None)
    if eos_id is None and hasattr(processor, "tokenizer"):
        eos_id = processor.tokenizer.eos_token_id

    eos_token_in_output = (eos_id is not None and eos_id in new_tokens.tolist())
    eos_tag_in_text = ("<end_of_turn>" in raw_response or "<eos>" in raw_response)
    eos_produced = bool(eos_token_in_output or eos_tag_in_text)
    hit_max_tokens = bool(num_gen >= max_new_tokens)

    return {
        "text": text,
        "num_gen_tokens": num_gen,
        "hit_max_tokens": hit_max_tokens,
        "eos_produced": eos_produced,
        "elapsed_s": elapsed,
        "tokens_per_s": num_gen / max(elapsed, 0.001),
    }

SYSTEM_PROMPTS = {
    "closed_book_knowledge": (
        "You are a careful Amharic writer. Answer only in Amharic (Ge'ez script).\n"
        "No English words, no markdown, no notes about your answer.\n"
        "Give a concise answer, at most 3-4 sentences.\n"
        "End with the Amharic full stop (።)."
    ),
    "instruction_following": (
        "You are a careful Amharic writer. Answer only in Amharic (Ge'ez script).\n"
        "No English words, no markdown, no notes about your answer.\n"
        "Give one short sentence taken from the passage.\n"
        "End with the Amharic full stop (።)."
    ),
    "summarization": (
        "You are a careful Amharic writer. Answer only in Amharic (Ge'ez script).\n"
        "No English words, no markdown, no notes about your answer.\n"
        "Give a 2-3 sentence summary.\n"
        "End with the Amharic full stop (።)."
    ),
    "translation": (
        "You are a careful translator.\n"
        "Output only the translation, nothing else.\n"
        "No notes, no markdown."
    ),
}

MAX_NEW_TOKENS_MAP = {
    "closed_book_knowledge": 300,
    "instruction_following": 200,
    "summarization": 300,
    "translation": 150,
}


In [ ]:
# TEACHER PILOT — HARD GATE (~100 prompts)

def geez_ratio(s):
    letters = [c for c in s if c.isalpha()]
    if not letters:
        return 0.0
    return sum(1 for c in letters if '\u1200' <= c <= '\u139f') / len(letters)

def latin_ratio(s):
    letters = [c for c in s if c.isalpha()]
    if not letters:
        return 0.0
    return sum(1 for c in letters if 'a' <= c.lower() <= 'z') / len(letters)

with open(PROJECT / 'data' / 'seed_prompts.json', encoding='utf-8') as f:
    seed_prompts = json.load(f)

# Sample 100 prompts from the 150 seeds
pilot_sample = random.sample(seed_prompts, min(100, len(seed_prompts)))

# Clear cache and reset peak memory tracking before pilot
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

TARGET_THROUGHPUT = 60.0
MINIMUM_THROUGHPUT = 42.0

pilot_results = []
total_tokens = 0
total_time = 0

print(f"Starting pilot on {len(pilot_sample)} seed prompts with {TEACHER_USED}...\n")

for i, p in enumerate(pilot_sample):
    sp = SYSTEM_PROMPTS.get(p['task'], SYSTEM_PROMPTS['closed_book_knowledge'])
    mt = MAX_NEW_TOKENS_MAP.get(p['task'], 300)
    try:
        res = generate_one(teacher_model, teacher_processor, p['prompt'], sp, max_new_tokens=mt)

        # Language-quality evaluation
        text = res['text']
        if p.get('direction') == 'am->en':
            lang_metric = {"script_type": "latin", "ratio": latin_ratio(text)}
        else:
            lang_metric = {"script_type": "geez", "ratio": geez_ratio(text)}

        res.update({
            "id": p["id"],
            "task": p["task"],
            "direction": p.get("direction", ""),
            "script_metric": lang_metric,
        })
        pilot_results.append(res)
        total_tokens += res['num_gen_tokens']
        total_time += res['elapsed_s']

        if (i + 1) % 10 == 0:
            current_tps = total_tokens / max(total_time, 0.001)
            cur_mem = torch.cuda.max_memory_allocated() / 1e9
            print(f"  [{i+1}/{len(pilot_sample)}] Throughput: {current_tps:.1f} tok/s | Allocated VRAM: {cur_mem:.2f} GB")
    except Exception as e:
        print(f"  Pilot error on {p['id']}: {e}")
        pilot_results.append({"id": p["id"], "error": str(e)})

# Memory measurements
peak_allocated_gb = torch.cuda.max_memory_allocated() / 1e9
peak_reserved_gb = torch.cuda.max_memory_reserved() / 1e9

ok_runs = [r for r in pilot_results if "error" not in r]
err_runs = [r for r in pilot_results if "error" in r]
actual_tps = total_tokens / max(total_time, 0.001)
truncation_count = sum(1 for r in ok_runs if r.get("hit_max_tokens"))

# Language quality breakdown
geez_evals = [r["script_metric"]["ratio"] for r in ok_runs if r["script_metric"]["script_type"] == "geez"]
latin_evals = [r["script_metric"]["ratio"] for r in ok_runs if r["script_metric"]["script_type"] == "latin"]

avg_geez = sum(geez_evals) / len(geez_evals) if geez_evals else 0.0
avg_latin = sum(latin_evals) / len(latin_evals) if latin_evals else 0.0

print(f"\n{'='*65}")
print("TEACHER PILOT REPORT")
print(f"{'='*65}")
print(f"  Teacher Model:          {TEACHER_USED}")
print(f"  Successful / Errors:    {len(ok_runs)} / {len(err_runs)}")
print(f"  Throughput:             {actual_tps:.1f} tokens/s (Target: {TARGET_THROUGHPUT}, Min: {MINIMUM_THROUGHPUT})")
print(f"  Peak Allocated VRAM:    {peak_allocated_gb:.2f} GB")
print(f"  Peak Reserved VRAM:     {peak_reserved_gb:.2f} GB")
print(f"  Truncated Generations:  {truncation_count} / {len(ok_runs)}")
print(f"  Avg Ge'ez Ratio (AM):   {avg_geez:.3f}")
print(f"  Avg Latin Ratio (EN):   {avg_latin:.3f}")
print(f"{'='*65}")

# Pilot Gate evaluation
passed_gate = (
    len(err_runs) <= 3 and
    actual_tps >= MINIMUM_THROUGHPUT and
    peak_reserved_gb <= 15.0
)

if passed_gate:
    print(f"\n>>> PILOT GATE PASSED: Continuing full run with {TEACHER_USED}")
else:
    print(f"\n>>> PILOT GATE FAILED: Performance or memory violation.")
    if TEACHER_USED != FALLBACK_ID:
        print(f"Switching immediately to {FALLBACK_ID}...")
        del teacher_model, teacher_processor
        gc.collect()
        torch.cuda.empty_cache()
        teacher_processor = AutoProcessor.from_pretrained(FALLBACK_ID)
        teacher_model = AutoModelForMultimodalLM.from_pretrained(
            FALLBACK_ID,
            quantization_config=bnb_config,
            device_map="auto",
            torch_dtype=torch.float16,
        )
        teacher_model.eval()
        TEACHER_USED = FALLBACK_ID
        print(f"Now running with fallback: {TEACHER_USED}")

with open(PROJECT / 'logs' / 'pilot_results.json', 'w', encoding='utf-8') as f:
    json.dump({
        "teacher": TEACHER_USED,
        "throughput_tps": actual_tps,
        "peak_allocated_gb": peak_allocated_gb,
        "peak_reserved_gb": peak_reserved_gb,
        "passed": passed_gate,
        "avg_geez_ratio": avg_geez,
        "avg_latin_ratio": avg_latin,
        "truncation_count": truncation_count,
    }, f, ensure_ascii=False, indent=2)

Starting pilot on 100 seed prompts with google/gemma-4-E4B-it...



KeyboardInterrupt: 

In [ ]:
from transformers import AutoTokenizer as AT
student_tok = AT.from_pretrained("google/gemma-3-1b-it")

fertility_sentences = [
    "ኢትዮጵያ ከአፍሪካ ሁለተኛ ሕዝብ ብዛት ያላት ሀገር ናት።",
    "አዲስ አበባ የአፍሪካ ህብረት ዋና መቀመጫ ናት።",
    "ቡና ኢትዮጵያ ውስጥ ለመጀመሪያ ጊዜ የተገኘ ተክል ነው።",
    "ትምህርት ለሁሉም ሰው መሠረታዊ መብት ነው።",
    "ንጹህ ውሃ ማግኘት ለጤና በጣም አስፈላጊ ነው።",
    "የኢትዮጵያ ብር ባለፉት ዓመታት ዋጋው ቀንሷል።",
    "ገበሬዎች ዘመናዊ የግብርና ቴክኖሎጂ እንዲጠቀሙ ማበረታታት ያስፈልጋል።",
    "በኢትዮጵያ ውስጥ ከ80 በላይ ብሄር ብሄረሰቦች ይኖራሉ።",
    "ፀሐይ ለምድራችን ብርሃንና ሙቀት ዋና ምንጭ ናት።",
    "ዛፎችን መትከል የአየር ንብረት ለውጥን ለመቀነስ ይረዳል።",
    "ወባ በትንኝ ንክሻ የሚተላለፍ በሽታ ነው።",
    "የኢትዮጵያ አየር መንገድ በአፍሪካ ትልቁ አየር መንገድ ነው።",
    "ስፖርት ለወጣቶች ጤንነትም ለአእምሮ እድገትም ይጠቅማል።",
    "ቴክኖሎጂ ትምህርትን ለሁሉም ተደራሽ ለማድረግ ያግዛል።",
    "እንጀራ ከጤፍ ዱቄት ይሠራል፤ ዋናው የኢትዮጵያ ምግብ ነው።",
    "ምርምር ማድረግ ለሳይንስ እና ለቴክኖሎጂ እድገት ወሳኝ ነው።",
    "ባህላዊ ሙዚቃ የህብረተሰቡን ታሪክና ማንነት ይገልጻል።",
    "ልጆች ደህንነታቸው የተጠበቀ ትምህርት ቤት ያስፈልጋቸዋል።",
    "የኢትዮጵያ ቅርስ ቦታዎች በዓለም ቅርስ ዝርዝር ውስጥ ይገኛሉ።",
    "ማንበብ ለአእምሮ እድገት እና ለእውቀት ማስፋት ይጠቅማል።",
]

print("Fertility Test (Gemma 3 1B-it tokenizer)\n" + "="*60)
tw, tt = 0, 0
for s in fertility_sentences:
    w = len(s.split())
    t = len(student_tok.encode(s, add_special_tokens=False))
    tw += w; tt += t
    print(f"  w={w:2d} t={t:3d} ratio={t/max(1,w):.2f} | {s[:50]}...")
print(f"\nAvg tokens/word: {tt/max(1,tw):.2f} | Vocab size: {len(student_tok)}")

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

Fertility Test (Gemma 3 1B-it tokenizer)
  w= 8 t= 24 ratio=3.00 | ኢትዮጵያ ከአፍሪካ ሁለተኛ ሕዝብ ብዛት ያላት ሀገር ናት።...
  w= 7 t= 17 ratio=2.43 | አዲስ አበባ የአፍሪካ ህብረት ዋና መቀመጫ ናት።...
  w= 8 t= 17 ratio=2.12 | ቡና ኢትዮጵያ ውስጥ ለመጀመሪያ ጊዜ የተገኘ ተክል ነው።...
  w= 6 t= 15 ratio=2.50 | ትምህርት ለሁሉም ሰው መሠረታዊ መብት ነው።...
  w= 7 t= 12 ratio=1.71 | ንጹህ ውሃ ማግኘት ለጤና በጣም አስፈላጊ ነው።...
  w= 6 t= 20 ratio=3.33 | የኢትዮጵያ ብር ባለፉት ዓመታት ዋጋው ቀንሷል።...
  w= 7 t= 26 ratio=3.71 | ገበሬዎች ዘመናዊ የግብርና ቴክኖሎጂ እንዲጠቀሙ ማበረታታት ያስፈልጋል።...
  w= 7 t= 23 ratio=3.29 | በኢትዮጵያ ውስጥ ከ80 በላይ ብሄር ብሄረሰቦች ይኖራሉ።...
  w= 7 t= 20 ratio=2.86 | ፀሐይ ለምድራችን ብርሃንና ሙቀት ዋና ምንጭ ናት።...
  w= 7 t= 21 ratio=3.00 | ዛፎችን መትከል የአየር ንብረት ለውጥን ለመቀነስ ይረዳል።...
  w= 6 t= 14 ratio=2.33 | ወባ በትንኝ ንክሻ የሚተላለፍ በሽታ ነው።...
  w= 8 t= 20 ratio=2.50 | የኢትዮጵያ አየር መንገድ በአፍሪካ ትልቁ አየር መንገድ ነው።...
  w= 6 t= 24 ratio=4.00 | ስፖርት ለወጣቶች ጤንነትም ለአእምሮ እድገትም ይጠቅማል።...
  w= 6 t= 20 ratio=3.33 | ቴክኖሎጂ ትምህርትን ለሁሉም ተደራሽ ለማድረግ ያግዛል።...
  w= 8 t= 24 ratio=3.00 | እንጀራ ከጤፍ ዱቄት ይሠራል፤ ዋናው የኢትዮጵያ ምግብ ነው።...
  w= 8 

In [ ]:
#  Generate Informational Amharic passages (~600): for this training, I manually stopped it at 300 ──────────
# Informational passages only (no fabricated news or current events).

INFORMATIONAL_TOPICS = [
    "Ethiopian history",
    "health and medicine",
    "science and technology",
    "agriculture and farming",
    "education",
    "environment and climate",
    "economy and trade",
    "culture and traditions",
    "geography",
    "infrastructure and development",
    "wildlife and conservation",
]

PASSAGE_SYS = (
    "You are a careful Amharic writer. Write only in Amharic (Ge'ez script).\n"
    "No English words, no markdown, no notes.\n"
    "Write an informational, educational passage.\n"
    "End with the Amharic full stop (።)."
)

all_passages = []
lengths_spec = [
    ("short", "400-600 characters"),
    ("medium", "600-900 characters"),
    ("long", "900-1200 characters"),
]

In [ ]:
print("Generating informational passages...")
for topic in INFORMATIONAL_TOPICS:
    for length_name, length_desc in lengths_spec:
        for _ in range(19):  # 11 topics * 3 lengths * 19 ≈ 627 passages
            user_msg = (
                f"Write an informational Amharic passage about {topic}.\n"
                f"Length: {length_desc}.\n"
                f"Write only the passage in Ge'ez script with facts and details."
            )
            try:
                r = generate_one(teacher_model, teacher_processor, user_msg, PASSAGE_SYS,
                                 max_new_tokens=256, temperature=0.6)
                all_passages.append({
                    "passage_id": f"pass_{len(all_passages):04d}",
                    "topic": topic,
                    "length_category": length_name,
                    "text": r["text"],
                    "character_length": len(r["text"]),  # Measure actual passage length
                    "num_gen_tokens": r["num_gen_tokens"],
                    "hit_max_tokens": r["hit_max_tokens"],
                    "eos_produced": r["eos_produced"],
                })
            except Exception as e:
                print(f"  Error generating passage for {topic}: {e}")

            if len(all_passages) % 100 == 0 and len(all_passages) > 0:
                shard_path = PROJECT / 'shards' / f'passages_{len(all_passages)//100}.json'
                with open(shard_path, 'w', encoding='utf-8') as f:
                    json.dump(all_passages[-100:], f, ensure_ascii=False, indent=2)
                print(f"  [{len(all_passages)}] passages generated (shard saved)")

            if len(all_passages) >= 600:
                break
        if len(all_passages) >= 600:
            break
    if len(all_passages) >= 600:
        break

with open(PROJECT / 'data' / 'passages_all.json', 'w', encoding='utf-8') as f:
    json.dump(all_passages, f, ensure_ascii=False, indent=2)
print(f"Total informational passages generated: {len(all_passages)}")

Generating informational passages...
  [100] passages generated (shard saved)
  [200] passages generated (shard saved)
  [300] passages generated (shard saved)


KeyboardInterrupt: 

In [ ]:
with open(PROJECT / 'data' / 'passages_all.json', 'w', encoding='utf-8') as f:
    json.dump(all_passages, f, ensure_ascii=False, indent=2)
print(f"Total informational passages generated: {len(all_passages)}")

NameError: name 'all_passages' is not defined

In [ ]:

# ── Cell 2.8: Expand prompts driven by all 150 handwritten seeds ──────

with open(PROJECT / 'data' / 'passages_all.json', encoding='utf-8') as f:
    all_passages = json.load(f)
with open(PROJECT / 'data' / 'seeds.json', encoding='utf-8') as f:
    seeds_data = json.load(f)

QSYS = (
    "You are a careful Amharic writer. Write only in Amharic (Ge'ez script).\n"
    "No English words, no markdown. End with the Amharic full stop (።)."
)

# 1. Expand Reading Comprehension (driven by 30 handwritten RC seeds)
rc_prompts = []
handwritten_rc = seeds_data["instruction_following"]

# Incorporate the 30 handwritten RC questions paired with initial passages
for i, q in enumerate(handwritten_rc):
    p_text = all_passages[i % len(all_passages)]["text"]
    rc_prompts.append({
        "passage_id": all_passages[i % len(all_passages)]["passage_id"],
        "passage": p_text,
        "question": q,
        "source": "handwritten_seed",
    })

# Teacher generates additional RC questions inspired by the style of the seeds
print(f"Expanding reading comprehension prompts (starting with {len(rc_prompts)} seeds)...")
for p in all_passages:
    if len(rc_prompts) >= 450:
        break
    seed_example = random.choice(handwritten_rc)
    user_prompt = (
        f"Read this Amharic passage and write ONE clear factual question answerable directly from it.\n"
        f"Example question style: {seed_example}\n"
        f"Write ONLY the question in Amharic:\n\nPassage:\n{p['text']}"
    )
    try:
        r = generate_one(teacher_model, teacher_processor, user_prompt, QSYS,
                         max_new_tokens=100, temperature=0.3)
        q_gen = r['text'].strip().split('\n')[0].strip()
        if len(q_gen) > 10:
            rc_prompts.append({
                "passage_id": p['passage_id'],
                "passage": p['text'],
                "question": q_gen,
                "source": "teacher_generated",
            })
    except Exception as e:
        print(f"  RC error on {p['passage_id']}: {e}")

# 2. Expand Summarization (driven by 30 handwritten summarization seeds)
sum_passages = list(seeds_data["summarization"])  # 30 handwritten passages
# Supplement with informational passages to reach at least 450
for p in all_passages:
    if len(sum_passages) >= 450:
        break
    sum_passages.append(p["text"])

# 3. Expand Closed-Book Knowledge (driven by 30 handwritten closed-book seeds)
expanded_cb = list(seeds_data["closed_book"])  # 30 handwritten seeds
print(f"Expanding closed-book questions (starting with {len(expanded_cb)} seeds)...")
for topic in INFORMATIONAL_TOPICS:
    seed_examples = "\n".join(random.sample(seeds_data["closed_book"], 3))
    user_prompt = (
        f"Generate 45 unique Amharic factual knowledge questions about {topic}.\n"
        f"Ge'ez script only. One question per line. No numbers or bullets.\n"
        f"Follow the style of these example questions:\n{seed_examples}"
    )
    try:
        r = generate_one(teacher_model, teacher_processor, user_prompt, QSYS,
                         max_new_tokens=800, temperature=0.7)
        new_q = [q.strip() for q in r['text'].split('\n')
                 if q.strip() and len(q.strip()) > 10 and not q.strip().startswith(('1','2','3','4','5'))]
        expanded_cb.extend(new_q)
        print(f"  {topic}: +{len(new_q)} (total {len(expanded_cb)})")
    except Exception as e:
        print(f"  CB expansion error on {topic}: {e}")

# Top-up closed book if needed
while len(expanded_cb) < 450:
    extra = f"የ{random.choice(INFORMATIONAL_TOPICS)} ዋና ዋና ጠቀሜታዎች ምንድናቸው?"
    expanded_cb.append(extra)

# 4. Expand Translation EN->AM (driven by 30 handwritten EN->AM seeds)
expanded_en = list(seeds_data["translation_en_am"])  # 30 handwritten seeds
print(f"Expanding EN->AM translation sentences (starting with {len(expanded_en)} seeds)...")
for topic in INFORMATIONAL_TOPICS:
    if len(expanded_en) >= 225:
        break
    user_prompt = (
        f"Write 25 diverse, informative English sentences about {topic}, 10 to 35 words each.\n"
        f"One sentence per line. No numbering."
    )
    try:
        r = generate_one(teacher_model, teacher_processor, user_prompt,
                         "You write informative English sentences. One per line.",
                         max_new_tokens=600, temperature=0.7)
        lines = [s.strip() for s in r['text'].split('\n') if len(s.strip().split()) >= 6]
        expanded_en.extend(lines)
    except Exception as e:
        print(f"  Translation EN expansion error: {e}")

# 5. Expand Translation AM->EN (driven by 30 handwritten AM->EN seeds)
expanded_am = list(seeds_data["translation_am_en"])  # 30 handwritten seeds
print(f"Expanding AM->EN translation sentences (starting with {len(expanded_am)} seeds)...")
for topic in INFORMATIONAL_TOPICS:
    if len(expanded_am) >= 225:
        break
    user_prompt = (
        f"ስለ {topic} 25 የተለያዩ አስተማሪ የአማርኛ ዓረፍተ ነገሮችን ፃፍ።\n"
        f"አንድ ዓረፍተ ነገር በአንድ መስመር ብቻ። ቁጥር አትጠቀም።"
    )
    try:
        r = generate_one(teacher_model, teacher_processor, user_prompt, QSYS,
                         max_new_tokens=600, temperature=0.7)
        lines = [s.strip() for s in r['text'].split('\n') if len(s.strip().split()) >= 4]
        expanded_am.extend(lines)
    except Exception as e:
        print(f"  Translation AM expansion error: {e}")



with open(PROJECT / 'data' / 'expanded_prompts.json', 'w', encoding='utf-8') as f:
    json.dump({
        "closed_book": expanded_cb,
        "rc_prompts": rc_prompts,
        "summarization_contexts": sum_passages,
        "translation_en": expanded_en,
        "translation_am": expanded_am,
    }, f, ensure_ascii=False, indent=2)

Expanding reading comprehension prompts (starting with 30 seeds)...
Expanding closed-book questions (starting with 30 seeds)...
  Ethiopian history: +34 (total 64)
  health and medicine: +43 (total 107)
  science and technology: +44 (total 151)
  agriculture and farming: +40 (total 191)
  education: +37 (total 228)
  environment and climate: +37 (total 265)
  economy and trade: +45 (total 310)
  culture and traditions: +30 (total 340)
  geography: +32 (total 372)
  infrastructure and development: +42 (total 414)
  wildlife and conservation: +35 (total 449)
Expanding EN->AM translation sentences (starting with 30 seeds)...
Expanding AM->EN translation sentences (starting with 30 seeds)...


In [ ]:
# ── Cell 2.9: Assemble exactly 1,800 raw prompt candidates ───────────

def format_prompt(instruction, context=None, question=None):
    parts = [instruction]
    if context:
        parts.append(f"\nContext:\n{context}")
    if question:
        parts.append(f"\n{question}")
    return "\n".join(parts)

with open(PROJECT / 'data' / 'expanded_prompts.json', encoding='utf-8') as f:
    exp = json.load(f)
with open(PROJECT / 'data' / 'seeds.json', encoding='utf-8') as f:
    seeds_data = json.load(f)

TEMPLATES = seeds_data['instruction_templates']
all_prompts = []
idx = 0

# 1. Exactly 450 Closed-book
for q in exp['closed_book'][:450]:
    instr = random.choice(TEMPLATES['closed_book_knowledge'])
    all_prompts.append({
        "id": f"p_{idx:04d}",
        "task": "closed_book_knowledge",
        "direction": "",
        "instruction": instr,
        "context": q,
        "question": "",
        "prompt": format_prompt(instr, context=q),
        "source": "seed_handwritten" if idx < 30 else "teacher_expanded",
    })
    idx += 1

# 2. Exactly 450 Reading Comprehension (Instruction Following)
for rc in exp['rc_prompts'][:450]:
    instr = random.choice(TEMPLATES['instruction_following'])
    all_prompts.append({
        "id": f"p_{idx:04d}",
        "task": "instruction_following",
        "direction": "",
        "instruction": instr,
        "context": rc['passage'],
        "question": rc['question'],
        "prompt": format_prompt(instr, context=rc['passage'], question=rc['question']),
        "source": rc.get('source', 'teacher_passage'),
    })
    idx += 1

# 3. Exactly 450 Summarization
for s_ctx in exp['summarization_contexts'][:450]:
    instr = random.choice(TEMPLATES['summarization'])
    all_prompts.append({
        "id": f"p_{idx:04d}",
        "task": "summarization",
        "direction": "",
        "instruction": instr,
        "context": s_ctx,
        "question": "",
        "prompt": format_prompt(instr, context=s_ctx),
        "source": "seed_handwritten" if (idx - 900) < 30 else "teacher_passage",
    })
    idx += 1

# 4. Exactly 225 Translation EN->AM
for sent in exp['translation_en'][:225]:
    instr = random.choice(TEMPLATES['translation_en_am'])
    all_prompts.append({
        "id": f"p_{idx:04d}",
        "task": "translation",
        "direction": "en->am",
        "instruction": instr,
        "context": sent,
        "question": "",
        "prompt": format_prompt(instr, context=sent),
        "source": "seed_handwritten" if (idx - 1350) < 30 else "teacher_expanded",
    })
    idx += 1

# 5. Exactly 225 Translation AM->EN
for sent in exp['translation_am'][:225]:
    instr = random.choice(TEMPLATES['translation_am_en'])
    all_prompts.append({
        "id": f"p_{idx:04d}",
        "task": "translation",
        "direction": "am->en",
        "instruction": instr,
        "context": sent,
        "question": "",
        "prompt": format_prompt(instr, context=sent),
        "source": "seed_handwritten" if (idx - 1575) < 30 else "teacher_expanded",
    })
    idx += 1

# ── Validation and Hard Assertions ───────────────────────────────────
cb_assembled = sum(1 for p in all_prompts if p['task'] == 'closed_book_knowledge')
rc_assembled = sum(1 for p in all_prompts if p['task'] == 'instruction_following')
sm_assembled = sum(1 for p in all_prompts if p['task'] == 'summarization')
en_assembled = sum(1 for p in all_prompts if p['task'] == 'translation' and p['direction'] == 'en->am')
am_assembled = sum(1 for p in all_prompts if p['task'] == 'translation' and p['direction'] == 'am->en')
total_assembled = len(all_prompts)

print("=" * 60)
print("ASSEMBLED CANDIDATE PROMPT DISTRIBUTION")
print("=" * 60)
print(f"  Closed-book:           {cb_assembled:4d}  (expected: 450)")
print(f"  Reading comprehension: {rc_assembled:4d}  (expected: 450)")
print(f"  Summarization:         {sm_assembled:4d}  (expected: 450)")
print(f"  Translation EN->AM:    {en_assembled:4d}  (expected: 225)")
print(f"  Translation AM->EN:    {am_assembled:4d}  (expected: 225)")
print("-" * 60)
print(f"  Total:                 {total_assembled:4d}  (expected: 1800)")
print("=" * 60)



with open(PROJECT / 'data' / 'all_prompts.json', 'w', encoding='utf-8') as f:
    json.dump(all_prompts, f, ensure_ascii=False, indent=2)
print("Saved all 1800 raw candidate prompts to all_prompts.json")
#I got some errors on the generated prompts so instead I used sources from HF datasets

ASSEMBLED CANDIDATE PROMPT DISTRIBUTION
  Closed-book:            450  (expected: 450)
  Reading comprehension:  349  (expected: 450)
  Summarization:          349  (expected: 450)
  Translation EN->AM:     225  (expected: 225)
  Translation AM->EN:     225  (expected: 225)
------------------------------------------------------------
  Total:                 1598  (expected: 1800)
Saved all 1800 raw candidate prompts to all_prompts.json


In [ ]:

# Teacher answers ALL prompts (resumable JSONL)

def load_prompts_safely(prompts_file):
    """Safely loads all prompts handling standard JSON, concatenated dumps (Extra data), or JSONL."""
    with open(prompts_file, 'r', encoding='utf-8', errors='replace') as f:
        content = f.read().strip()
    try:
        data = json.loads(content)
        if isinstance(data, list):
            return data
    except json.JSONDecodeError:
        pass

    # Handle multiple JSON dumps or concatenated arrays (Extra data error)
    items = []
    decoder = json.JSONDecoder()
    idx, length = 0, len(content)
    while idx < length:
        while idx < length and content[idx].isspace():
            idx += 1
        if idx >= length:
            break
        try:
            obj, end = decoder.raw_decode(content, idx)
            if isinstance(obj, list):
                items.extend(obj)
            elif isinstance(obj, dict):
                items.append(obj)
            idx = end
        except json.JSONDecodeError:
            break

    if items:
        seen, deduped = set(), []
        for x in items:
            iid = x.get('id') if isinstance(x, dict) else None
            if iid and iid in seen:
                continue
            if iid:
                seen.add(iid)
            deduped.append(x)
        return deduped

    # Fallback to line-by-line JSONL
    line_items = []
    for line in content.splitlines():
        line = line.strip()
        if line:
            try:
                line_items.append(json.loads(line))
            except:
                pass
    if line_items:
        return line_items
    raise ValueError(f"Could not parse valid JSON from {prompts_file}")

prompts_path = PROJECT / 'data' / 'all_prompts.json'
if not prompts_path.exists() and (PROJECT / 'data' / 'all_prompts_hf.json').exists():
    prompts_path = PROJECT / 'data' / 'all_prompts_hf.json'

all_prompts = load_prompts_safely(prompts_path)

# Ensure the file is saved back in clean, valid JSON format
with open(PROJECT / 'data' / 'all_prompts.json', 'w', encoding='utf-8') as f:
    json.dump(all_prompts, f, ensure_ascii=False, indent=2)
print(f"Loaded and verified {len(all_prompts)} prompts from {prompts_path.name}")

answers_path = PROJECT / 'data' / 'teacher_answers_raw.jsonl'
done_ids = set()
if answers_path.exists():
    with open(answers_path, encoding='utf-8') as f:
        for line in f:
            if line.strip():
                try:
                    obj = json.loads(line)
                    done_ids.add(obj.get('id'))
                except:
                    pass
    print(f"Resuming generation: {len(done_ids)} prompts already completed.")

remaining = [p for p in all_prompts if p['id'] not in done_ids]
# Sort by prompt length to improve batch predictability and memory efficiency
remaining.sort(key=lambda p: len(p['prompt']))
print(f"Prompts remaining to generate: {len(remaining)}")

start_time = time.time()
completed = 0

with open(answers_path, 'a', encoding='utf-8') as fout:
    for i, p in enumerate(remaining):
        sp = SYSTEM_PROMPTS.get(p['task'], SYSTEM_PROMPTS['closed_book_knowledge'])
        mt = MAX_NEW_TOKENS_MAP.get(p['task'], 300)
        try:
            r = generate_one(teacher_model, teacher_processor, p['prompt'], sp, max_new_tokens=mt)
            record = {
                "id": p['id'],
                "task": p['task'],
                "direction": p.get('direction', ''),
                "prompt": p['prompt'],
                "instruction": p['instruction'],
                "context": p.get('context', ''),
                "question": p.get('question', ''),
                "response": r['text'],
                "num_generated_tokens": r['num_gen_tokens'],
                "hit_max_tokens": r['hit_max_tokens'],
                "eos_produced": r['eos_produced'],
                "teacher_id": TEACHER_USED,
                "source": p['source'],
            }
            fout.write(json.dumps(record, ensure_ascii=False) + '\n')
            fout.flush()
            completed += 1
        except Exception as e:
            print(f"  Error on prompt {p['id']}: {e}")
            fout.write(json.dumps({"id": p['id'], "error": str(e)}, ensure_ascii=False) + '\n')
            fout.flush()

        if (i + 1) % 50 == 0:
            elapsed_min = (time.time() - start_time) / 60
            vram_gb = torch.cuda.max_memory_allocated() / 1e9
            print(f"  [{i+1}/{len(remaining)}] {completed} answers saved | {elapsed_min:.1f} min elapsed | VRAM: {vram_gb:.2f} GB")

print(f"\nTeacher generation complete: {completed} answers recorded in {(time.time()-start_time)/60:.1f} min")



Loaded and verified 1500 prompts from all_prompts.json
Resuming generation: 1157 prompts already completed.
Prompts remaining to generate: 342
  [50/342] 50 answers saved | 5.7 min elapsed | VRAM: 9.54 GB
  [100/342] 100 answers saved | 9.8 min elapsed | VRAM: 9.54 GB
  [150/342] 150 answers saved | 13.7 min elapsed | VRAM: 9.56 GB
  [200/342] 200 answers saved | 18.2 min elapsed | VRAM: 9.58 GB
  [250/342] 250 answers saved | 22.4 min elapsed | VRAM: 9.58 GB
  [300/342] 300 answers saved | 26.5 min elapsed | VRAM: 9.63 GB

Teacher generation complete: 342 answers recorded in 30.6 min
